**Table of contents**<a id='toc0_'></a>    
- 1. [Notebook initialization](#toc1_)    
- 2. [Classes to manage XDF files](#toc2_)    
- 3. [Kinect stream processing](#toc3_)    
  - 3.1. [Make the kinect time correction (if needed)](#toc3_1_)    
  - 3.2. [Remove all rows filled with only zeros in the kinect data](#toc3_2_)    
  - 3.3. [Interpolate the Mocap data](#toc3_3_)    
  - 3.4. [Low pass filter](#toc3_4_)    
- 4. [Get the action zones from the ToNIC markers](#toc4_)    
- 5. [Get the conditions](#toc5_)    
  - 5.1. [Get the reaches](#toc5_1_)    
  - 5.2. [Get the conditions from the reaches sequence](#toc5_2_)    
- 6. [Get target position](#toc6_)    
- 7. [Compute the distance to the target for both wrists and both shoulders](#toc7_)    
- 8. [Re-compute the reaches start and stop times from the distance to the target data](#toc8_)    
- 9. [Save the reaches in a csv file](#toc9_)    
- 10. [Compute the panu ](#toc10_)    
- 11. [plot the reaches used to compute the panu](#toc11_)    
- 12. [Compute panu for one xdf file](#toc12_)    
- 13. [Compute panu for one visit](#toc13_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[Notebook initialization](#toc0_)

In [ ]:
import pyxdf
import numpy as np
import pandas as pd
import os
import logging
from PIL import Image
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, find_peaks


# and set to true for testing (default) but can be already set to false from outside (e.g., by the test script)
if "doRunTests" not in globals():
    doRunTests = True

if doRunTests:
    # the best for debug-test plots (external window that you can make fullscreen and zoom)
    %matplotlib qt


    # functions for plotting the tests... 
    def plot_before_after(time_before, data_before, time_after, data_after, title_txt=""):
        """
        Plot the data before and after doing some changes.
        """
        plt.figure()
        plt.plot(time_before, data_before, "+-", label="before")
        plt.plot(time_after, data_after, "*", label="after")
        plt.title(f"Data: before and after {title_txt}")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        plt.legend()
        plt.show()

# 2. <a id='toc2_'></a>[Classes to manage XDF files](#toc0_)

In [ ]:
class XDF_file:
    """
    Class to handle XDF files.
    """

    def __init__(
        self,
        xdf_fullFname: str | os.PathLike,
        select_streams=None,
    ):
        """
        Initialize the XDF file.
        Parameters
        ----------
        xdf_fullFname : str
            Full path to the XDF file.
        select_streams : list of str (as in load_xdf)
            List of stream types to select. If None, all streams are selected.
        """
        self.xdf_fullFname = xdf_fullFname
        self.xdf_fname = os.path.basename(xdf_fullFname)
        self.xdf_dir = os.path.dirname(xdf_fullFname)
        xdf_streams, xdf_header = pyxdf.load_xdf(
            xdf_fullFname,
            select_streams=select_streams,
            synchronize_clocks=True,
            dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
            verbose=False,
        )

        self.streams = []
        for i in range(len(xdf_streams)):
            stream = xdf_streams[i]
            self.streams.append(XDF_stream(stream))

    def __str__(self):
        """Print the names and types of all streams in the xdf file"""
        s = ""
        for i in range(len(self.streams)):
            stream = self.streams[i]
            s_type = stream.type
            s_name = stream.name
            s += f"Stream {i}: {s_type}, {s_name}\n"
        return s

    def get_stream_index(self, searched_stream_type, searched_stream_names):
        """Get the index of the stream of type 'searched_stream_type' AND with name in
        'searched_stream_names'"""

        if not isinstance(searched_stream_names, list):
            # if we get a string (only one name)
            searched_stream_names = [searched_stream_names]

        found_streams = []
        for i, stream in enumerate(self.streams):
            if searched_stream_type == stream.type:
                for searched_stream_name in searched_stream_names:
                    if searched_stream_name == stream.name:
                        found_streams.append(i)

        if not found_streams:
            return None

        if len(found_streams) > 1:
            found_streams_names = [self.streams[i].name for i in found_streams]
            msg = (
                f"Found multiple streams: "
                f"[{searched_stream_type},{found_streams_names}]."
            )
            raise ValueError(msg)

        return found_streams[0]


class XDF_channel:
    """
    Class to handle XDF channels.
    """

    def __init__(self, index, stream):
        self.index = index
        desc = stream["info"]["desc"][0]["channels"][0]["channel"][index]
        self.label = desc["label"][0]
        self.type = desc["type"][0]
        self.unit = desc["unit"][0]

        self.time_series = stream["time_series"][:, index]
        self.time_stamps = stream["time_stamps"]

    def __str__(self):
        """Print the name and type of the channel"""
        s = f"Channel {self.index}: {self.label} ({self.type},  {self.unit})\n"
        return s


class XDF_stream:
    """
    Class to handle XDF streams.
    A stream can be organized by channel (stream.channels = [...],  e.g. for data) or
    not organized (stream.channels = [], e.g. for markers ).
    """

    def __init__(self, xdf_stream):
        self.xdf_stream = xdf_stream
        self.time_stamps = xdf_stream["time_stamps"]
        self.time_series = xdf_stream["time_series"]
        self.name = xdf_stream["info"]["name"][0]
        self.type = xdf_stream["info"]["type"][0]
        self.channels = self.set_channels()

    def __str__(self):
        """Print the names and types of all channels in the stream"""
        s = f"Stream {self.name} ({self.type})\n"
        for i in range(len(self.channels)):
            channel_name = self.channels[i].label
            channel_type = self.channels[i].type
            channel_unit = self.channels[i].unit
            s += f"Channel {i}: {channel_name} ({channel_type}, {channel_unit})\n"
        return s

    def set_channels(self):
        """Set the channels from the xdf stream"""
        channels = []
        try:
            n_channels = len(
                self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"]
            )
            for i in range(n_channels):
                channel = XDF_channel(i, self.xdf_stream)
                channels.append(channel)
        except Exception:
            pass

        return channels

    def get_one_channel(self, name):
        for i in range(len(self.channels)):
            channel = self.channels[i]
            if channel.label == name:
                return channel
        return None

    def get_channel_index(self, channel_name):
        """Get the index of one channel from the stream by its name"""
        channel_index = -1
        nb_channels = len(self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"])
        for i in range(nb_channels):
            current_name = self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"][
                i
            ]["label"][0]
            if current_name == channel_name:
                channel_index = i
                break
        if channel_index == -1:
            return None

        return channel_index


def interpolate_to_constant_time_step(t, x, dt=1 / 30):
    """Interpolate the data to a constant time step"""

    n_columns = x.shape[1] if x.ndim > 1 else 1

    t_new = np.arange(t[0], t[-1], dt)

    if x.ndim < 2:
        x_new = np.interp(t_new, t, x)
    else:
        x_new = np.zeros((len(t_new), n_columns))
        for i in range(n_columns):
            x_new[:, i] = np.interp(t_new, t, x[:, i])

    return x_new, t_new

In [ ]:
if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf"  # no mouse marker csv file associated -> nan time correction - solution found
    xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P38/ReArm_C1P38_20231009_V1/ReArm_C1P38_20231009_V1_Reaching/ReArm_C1P38_20231009_V1_r.xdf"  # Test file for panu identification

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    print(xdf_file)
    i_kinect = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_kinect:
        print(xdf_file.streams[i_kinect])  # print the stream = all channels
        print(xdf_file.streams[i_kinect].channels[0])
        print(xdf_file.streams[i_kinect].get_one_channel("SpineBase_X"))

# 3. <a id='toc3_'></a>[Kinect stream processing](#toc0_)

## 3.1. <a id='toc3_1_'></a>[Make the kinect time correction (if needed)](#toc0_)

Any `*.xdf` file needing a correction of the kinect timestamps is accompanied by a `*_xdf_time_correction.csv` file with the time correction. 
If no such a file is present, no correction is needed.

In [ ]:
def get_time_correction(xdf_file: XDF_file):
    """Get the time correction from the xdf file name"""

    xdf_fullFname = str(xdf_file.xdf_fullFname)
    is_log_present = False
    log_fullFname = os.path.join(
        os.path.dirname(xdf_fullFname), "..", "kinect_time_correction.log"
    )
    log_fullFname = os.path.normpath(log_fullFname)
    if os.path.exists(log_fullFname):
        is_log_present = True

    is_correction_present = False
    time_correction_file = xdf_fullFname.replace(".xdf", "_xdf_time_correction.csv")
    if os.path.exists(time_correction_file):
        is_correction_present = True

    if not is_log_present:
        raise FileNotFoundError(
            f"The file {log_fullFname} does not exist. Please run the script to generate it."
        )

    if not is_correction_present:
        time_correction = np.float64(0)
    else:
        time_correction = np.loadtxt(time_correction_file, delimiter=",", skiprows=1)

    return time_correction


def make_kinect_time_correction(xdf_file: XDF_file):
    """Make the time correction for the Kinect streams in the xdf file."""

    # get the time correction from the xdf file name
    time_correction = get_time_correction(xdf_file)

    if time_correction == 0:
        # no time correction needed
        return

    if np.isnan(time_correction):
        # this can happen when the mouse csv file is not present...
        logging.warning(
            f"Time correction is NaN. Is there a mouse csv file for {xdf_file.xdf_fullFname}?"
        )
        return

    # here, we are OK to make the time correction
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    i_k_mk = xdf_file.get_stream_index("Markers", "EuroMov-Markers-Kinect")

    if i_k_mo is None or i_k_mk is None:
        raise ValueError(
            f"Cannot find the Kinect streams in {xdf_file.xdf_fullFname}. "
            f"Please check the stream names."
        )

    xdf_file.streams[i_k_mo].time_stamps += time_correction
    xdf_file.streams[i_k_mk].time_stamps += time_correction
    logging.info(
        f"Time correction of {time_correction} s applied to the Kinect streams."
    )

## 3.2. <a id='toc3_2_'></a>[Remove all rows filled with only zeros in the kinect data](#toc0_)

In [ ]:
def remove_zero_rows(xdf_file: XDF_file):
    """
    Remove the rows filled with zeros from the kinect data.
    """

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")

    if i_k_mo:
        kinect_t = xdf_file.streams[i_k_mo].time_stamps
        kinect_data = xdf_file.streams[i_k_mo].time_series
        nb_rows, nb_columns = kinect_data.shape

        # find the indexes of kinect data that are filled with zeros
        zero_rows = np.all(kinect_data == 0, axis=1)
        zero_rows_indices = np.where(zero_rows)[0]

        if len(zero_rows_indices) > 0:
            # remove the zero rows from the data
            kinect_data = np.delete(kinect_data, zero_rows_indices, axis=0)
            kinect_t = np.delete(kinect_t, zero_rows_indices, axis=0)

            # modify the stream (but not the original xdf_file.xdf_streams )
            xdf_file.streams[i_k_mo].time_series = kinect_data
            xdf_file.streams[i_k_mo].time_stamps = kinect_t

            logging.warning(
                f"Removed {len(zero_rows_indices)}/{nb_rows} rows filled with only zeros in the Kinect data"
            )


if doRunTests:

    # test the function
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution
    # xdf_fullFname =  '../dat/ReArm.lnk/DATA_named/C1P03/V2/Reaching/003_DucPas_20210430_2_r.xdf' # a lot of empty values

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo is None:
        raise ValueError("No MoCap stream found")

    i_wz = xdf_file.streams[i_k_mo].get_channel_index("WristRight_Z")

    t_before = xdf_file.streams[i_k_mo].time_stamps.copy()
    data_before = xdf_file.streams[i_k_mo].time_series[:, i_wz].copy()

    remove_zero_rows(xdf_file)

    t_after = xdf_file.streams[i_k_mo].time_stamps
    data_after = xdf_file.streams[i_k_mo].time_series[:, i_wz]

    plot_before_after(t_before, data_before, t_after, data_after)


## 3.3. <a id='toc3_3_'></a>[Interpolate the Mocap data](#toc0_)
This is mandatory because the kinect and mouse data are produced by the computer: the sampling rate is not waranted to be constant (samples are forgetten... sometimes). 

In [ ]:
def resample_kinect_data(xdf_file: XDF_file):
    """
    Resample the Kinect data to a constant time step.
    """

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo:
        kinect_t = xdf_file.streams[i_k_mo].time_stamps
        kinect_data = xdf_file.streams[i_k_mo].time_series

        initial_sampling_rate = 1 / (kinect_t[1] - kinect_t[0])

        # resample the data to a constant time step
        time_step = 1 / 30
        kinect_data, kinect_t = interpolate_to_constant_time_step(
            kinect_t, kinect_data, dt=time_step
        )

        # modify the stream
        xdf_file.streams[i_k_mo].time_series = kinect_data
        xdf_file.streams[i_k_mo].time_stamps = kinect_t

        logging.info(
            f"Kinect resampled at {1/time_step:3.2f} Hz (from about {initial_sampling_rate:3.2f} Hz before)."
        )


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution
    # NOTE: the sampling rate is 15hz in this xdf file

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")

    if i_k_mo:
        remove_zero_rows(xdf_file)

        stream = xdf_file.streams[i_k_mo]
        i_wz = stream.get_channel_index("WristRight_Z")
        if i_wz is None:
            raise ValueError("No WristRight_Z channel found")
        t_before = stream.time_stamps.copy()
        data_before = stream.time_series[:, i_wz].copy()

        resample_kinect_data(xdf_file)

        t_after = stream.time_stamps
        data_after = stream.time_series[:, i_wz]
        plot_before_after(t_before, data_before, t_after, data_after)

## 3.4. <a id='toc3_4_'></a>[Low pass filter](#toc0_)

This will be used later to find the reaches.

In [ ]:
def butter_lowpass(cutoff, fs, order=2):
    """Design a lowpass Butterworth filter."""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    sos = butter(
        order,
        normal_cutoff,
        btype="low",
        output="sos",  # recommended for numerical stability by scipy
    )
    return sos


def lowpass_filter(t, data, cutoff=0.5):
    """Apply a lowpass filter to the data"""
    # check that the sampling period is constant
    dt = np.mean(np.diff(t))
    if not np.allclose(np.diff(t), dt):
        raise ValueError("The time vector do not have a constant sampling period")

    fs = 1 / dt  # sample rate, Hz
    order = 4  # order of the filter
    sos = butter_lowpass(cutoff, fs, order=order)
    filtered_data = sosfiltfilt(sos, data)
    return filtered_data


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")

    if i_k_mo:
        remove_zero_rows(xdf_file)
        resample_kinect_data(xdf_file)

        stream = xdf_file.streams[i_k_mo]
        i_wz = stream.get_channel_index("WristRight_Z")
        if i_wz is None:
            raise ValueError("No WristRight_Z channel found")
        t_before = stream.time_stamps.copy()
        data_before = stream.time_series[:, i_wz].copy()

        filtered_data = lowpass_filter(t_before, data_before, cutoff=0.5)

        plot_before_after(t_before, data_before, t_before, filtered_data, "filter")

# 4. <a id='toc4_'></a>[Get the action zones from the ToNIC markers](#toc0_)


Each action zone has a *start* and *end* marker, but the label of the markers differs if the sequence was generated by : 

- the software **LSL-Mouse**: stream  `"Markers", ["MouseToNIC"]`
    - start = `"[111]"`  
    - stop = `"[100]"` 

- the software **event-IDE**: stream  `"Markers", ["event_ide_TONIC"]`
    - start = `"[100]"`, but we have to keep only the first start in the sequence before each stop
    - stop = `"[75]"`



In [ ]:
def find_marker_indexes(marker_name: str, markers_data: list):
    """Find the indexes of marker_name in markers_data"""

    marker_index_list = [
        i for i, marker in enumerate(markers_data) if marker_name in marker[0]
    ]

    return np.array(marker_index_list)


def get_coherent_actions_start_stop_times(starts, stops):
    """Get the coherent start and stop times of the actions"""

    # NOTE: we need this because of bugs in the data acquisition...

    # make an array of start, 0 and stop, 1
    start = np.zeros(
        len(starts),
        dtype=[("time", float), ("type", int)],
    )
    start["time"] = starts
    start["type"] = 0
    stop = np.zeros(
        len(stops),
        dtype=[("time", float), ("type", int)],
    )
    stop["time"] = stops
    stop["type"] = 1
    start_and_stop = np.concatenate((start, stop))
    start_and_stop = np.sort(start_and_stop, order="time")

    for i in range(len(start_and_stop) - 1):
        # keep only the last start in case of multiple contiguous start
        if start_and_stop[i]["type"] == 0 and start_and_stop[i + 1]["type"] == 0:
            start_and_stop[i + 1]["type"] = -1
        # keep only the first stop in case of multiple contiguous stop
        if start_and_stop[i]["type"] == 1 and start_and_stop[i + 1]["type"] == 1:
            start_and_stop[i + 1]["type"] = -1

    # ensure that the first is a start and the last is a stop
    if start_and_stop[0]["type"] == 1:
        start_and_stop[0]["type"] = -1
    if start_and_stop[-1]["type"] == 0:
        start_and_stop[-1]["type"] = -1

    # clean the start_and_stop array
    start_and_stop_ok = start_and_stop[start_and_stop["type"] != -1]

    starts_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 0]
    stops_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 1]

    if len(starts_corrected) != len(stops_corrected):
        raise ValueError(
            f"Number of starts ({len(starts_corrected)}) and stops ({len(stops_corrected)}) are not equal"
        )

    #### remove actions that are too short (less than 5s)
    actions_durations = stops_corrected - starts_corrected
    too_short_actions = actions_durations < 5
    if np.any(too_short_actions):
        logging.warning(
            f"Removing {np.sum(too_short_actions)} actions that are too short (< 5s)"
        )
        starts_corrected = starts_corrected[~too_short_actions]
        stops_corrected = stops_corrected[~too_short_actions]

    return starts_corrected, stops_corrected


def get_actions_start_stop_times_from_mouse_to_nic_markers(mouse_to_nic_markers):
    """get the start and stop times from mouse_to_nic_markers"""

    mouse_to_nic_markers_data = mouse_to_nic_markers.time_series
    mouse_to_nic_markers_time = mouse_to_nic_markers.time_stamps

    if not isinstance(mouse_to_nic_markers_data[0], list):
        mouse_to_nic_markers_data = [[str(x)] for x in mouse_to_nic_markers_data]

    start_marker_index_list = find_marker_indexes("[111]", mouse_to_nic_markers_data)
    stop_marker_index_list = find_marker_indexes("[100]", mouse_to_nic_markers_data)
    start_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        start_marker_index_list
    ]
    stop_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        stop_marker_index_list
    ]

    start_times_from_mouse_to_nic_markers, stop_times_from_mouse_to_nic_markers = (
        get_coherent_actions_start_stop_times(
            start_times_from_mouse_to_nic_markers,
            stop_times_from_mouse_to_nic_markers,
        )
    )

    return (
        start_times_from_mouse_to_nic_markers,
        stop_times_from_mouse_to_nic_markers,
    )


def get_actions_start_stop_times_from_event_ide_TONIC(event_ide_tonic):
    """get the start and stop times from event_ide_tonic"""

    event_ide_markers_data = event_ide_tonic.time_series
    event_ide_markers_time = event_ide_tonic.time_stamps

    if not isinstance(event_ide_markers_data[0], list):
        event_ide_markers_data = [[str(x)] for x in event_ide_markers_data]

    start_marker_index_list = find_marker_indexes("[100]", event_ide_markers_data)
    stop_marker_index_list = find_marker_indexes("[75]", event_ide_markers_data)

    start_times_from_event_ide_tonic = event_ide_markers_time[start_marker_index_list]
    stop_times_from_event_ide_tonic = event_ide_markers_time[stop_marker_index_list]

    # keep only the first start time for each stop time
    previous_stop_time = 0
    good_start_times = []
    for stop_time in stop_times_from_event_ide_tonic:
        possible_start_times = start_times_from_event_ide_tonic[
            start_times_from_event_ide_tonic < stop_time
        ]
        possible_start_times = possible_start_times[
            possible_start_times > previous_stop_time
        ]
        start_time = possible_start_times[0] if len(possible_start_times) > 0 else None
        good_start_times.append(start_time)
        previous_stop_time = stop_time
    good_start_times = np.array(good_start_times)
    good_start_times = good_start_times[
        good_start_times != None  # noqa: E711
    ]  # should be useless...

    good_start_times, stop_times_from_event_ide_tonic = (
        get_coherent_actions_start_stop_times(
            good_start_times,
            stop_times_from_event_ide_tonic,
        )
    )

    return (
        good_start_times,
        stop_times_from_event_ide_tonic,
    )


def get_actions_from_markers(xdf_file: XDF_file):
    """Get the start and stop times of the actions from the markers"""

    i_e2n = xdf_file.get_stream_index("Markers", ["event_ide_TONIC"])
    i_m2n = xdf_file.get_stream_index("Markers", ["MouseToNIC"])

    if i_e2n:
        start_t, stop_t = get_actions_start_stop_times_from_event_ide_TONIC(
            xdf_file.streams[i_e2n]
        )
        stream_name = xdf_file.streams[i_e2n].name
    elif i_m2n:
        start_t, stop_t = get_actions_start_stop_times_from_mouse_to_nic_markers(
            xdf_file.streams[i_m2n]
        )
        stream_name = xdf_file.streams[i_m2n].name
    else:
        raise ValueError("No event_ide_TONIC or MouseToNIC stream found")

    return {
        "start_times": start_t,
        "stop_times": stop_t,
        "stream_name": stream_name,
    }


def print_actions_start_stop_times(start_stop_times):
    """Print the start and stop times of the actions"""

    start_times = start_stop_times["start_times"]
    stop_times = start_stop_times["stop_times"]

    print(f"Stream {start_stop_times['stream_name']}: ")
    for i in range(len(start_times)):
        print(
            f"action{i:02d}: {start_times[i]:8.2f} -> {stop_times[i]:8.2f}, Duration: {stop_times[i] - start_times[i]:5.2f}s"
        )


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P05/V1/Reaching/C01P05_EttMha_20210621_1_r.xdf"  # no mouse marker csv file associated -> nan time correction - solution found

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)
    print(xdf_file)

    actions = get_actions_from_markers(xdf_file)
    print_actions_start_stop_times(actions)

    # plot the actions
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo:
        stream = xdf_file.streams[i_k_mo]
        t = stream.time_stamps
        i_wz = stream.get_channel_index("WristRight_Z")
        i_wl = stream.get_channel_index("WristLeft_Z")
        if i_wz is None or i_wl is None:
            raise ValueError("No WristRight_Z or WristLeft_Z channel found")
        wrist_r_z_data = stream.time_series[:, i_wz]
        wrist_l_z_data = stream.time_series[:, i_wl]

        plt.figure()
        plt.plot(t, wrist_r_z_data, "ok", label="WristRight_Z")
        plt.plot(t, wrist_l_z_data, "ob", label="WristLeft_Z")
        plt.title(f"Actions from markers: {actions['stream_name']}")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        for i in range(len(actions["start_times"])):
            plt.axvline(
                x=actions["start_times"][i],
                color="g",
                linestyle="--",
                # label=f"action{i:02d} start",
            )
            plt.axvline(
                x=actions["stop_times"][i],
                color="r",
                linestyle="--",
                # label=f"action{i:02d} stop",
            )
        plt.legend()
        plt.show()

# 5. <a id='toc5_'></a>[Get the conditions](#toc0_)

Here, we infer the sau-mau-cal conditions from the reaches sequence.

In the ReArm protocol, the participant is asked to follow a timed sequence of Action-Pause (20s each) in which the participant is asked to perform a series of 5 reaches (Action) and then wait for the next action (Pause).
In each condition, the participant performs 3 repetitions of Action-Pause (3 times 20+20s =  120s).

The conditions occur in the following order:     
1. **sau**: spontaneous reach sequence, first with the paretic hand, then with the non-paretic hand
2. **mau**: maximal reach sequence, first with the non-paretic hand, then with the paretic hand

There is another condition, which can either happen before or after the sau-mau sequence, to indicate where is the target zone (calibration): 

- **cal**: first with the paretic hand, then with the non-paretic hand (only one reach per hand, 1s stay in the target zone)

## 5.1. <a id='toc5_1_'></a>[Get the reaches](#toc0_)

In [ ]:
def get_reaches_on_one_wrist(t, wrist, wrist_f):
    """get the reaches on this wrist"""

    peaks_threshold = np.median(wrist_f) - 0.1
    i_inter_peaks = 60  # 2 seconds

    # find the negative peaks in the wrist that prominent of 0.1 and at least 2 seconds apart
    # NOTE: prominence works very well but it is difficult to understand the meaning of the value
    # neg_peaks, _ = find_peaks(-wrist_f, prominence=0.1, distance=60)  # 60 = 2s

    # find the negative peaks in the wrist that are below the threshold and at least 2 seconds apart
    # NOTE: this is the simplest method to find the peaks (same as method 1)
    neg_peaks, _ = find_peaks(-wrist_f, height=-peaks_threshold, distance=i_inter_peaks)

    # remove the peaks that are outliers (all reaches should end at the same target position)
    reaches_end = wrist_f[neg_peaks]
    reaches_end_median = np.median(reaches_end)
    reaches_end_iqr = np.percentile(reaches_end, 75) - np.percentile(reaches_end, 25)
    i_outliers = [
        i
        for i in range(len(reaches_end))
        if abs(reaches_end[i] - reaches_end_median) > 3 * reaches_end_iqr
    ]
    neg_peaks = np.delete(neg_peaks, i_outliers)

    # equivalent to index_of_last_negative_velocity_before_peak()... but simpler conceptually
    pos_peaks, _ = find_peaks(wrist_f)

    # we should start with a positive peak... 
    # TODO: not sure this makes sense, maybe better to suppress. 
    if len(pos_peaks) > 0 and pos_peaks[0] > neg_peaks[0]:
        # add a positive peak at the beginning
        pos_peaks = np.insert(pos_peaks, 0, 0)

    # put negative and positive peaks in the same list sorted by time
    negative_peaks = [
        {"index": neg_peaks[i], "from": "neg"} for i in range(len(neg_peaks))
    ]
    positive_peaks = [
        {"index": pos_peaks[i], "from": "pos"} for i in range(len(pos_peaks))
    ]
    pks = negative_peaks + positive_peaks
    pks = sorted(pks, key=lambda x: x["index"])

    # for each negative peak, keep only the previous positive peak
    reaches = []
    for i in range(1, len(pks)):
        if pks[i]["from"] == "neg":
            i_end = pks[i]["index"]
            i_beg = pks[i - 1]["index"]
            reach_distance = wrist_f[i_end] - wrist_f[i_beg]
            if -reach_distance > 0.05:  # 5 cm
                reaches.append(
                    {
                        "i_beg": i_beg,
                        "i_end": i_end,
                        "t_beg": t[i_beg],
                        "t_end": t[i_end],
                        "length": reach_distance,
                        "beg_position": wrist_f[i_beg],
                        "end_position": wrist_f[i_end],
                    }
                )

    def get_i_outliers(dimension):
        """Get the indexes of the outliers in the reaches length"""
        dimension_median = np.median(dimension)
        dimension_iqr = np.percentile(dimension, 75) - np.percentile(dimension, 25)
        i_outliers = [
            i
            for i in range(len(dimension))
            if abs(dimension[i] - dimension_median) > 3 * dimension_iqr
        ]
        return i_outliers
    

    # # remove the peaks that are large outliers 
    # # NOTE: it is best to remove the outliers (they are outlier!)
    reaches_length = np.array([r["length"] for r in reaches])
    i_out_reaches_length = get_i_outliers(reaches_length)
    reaches = np.delete(reaches, i_out_reaches_length)
    # length outliers might happen in the first calib reach... see
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P02/V2/Reaching/002_CorJea_20210409_2_r.xdf"

    reaches_end_position = np.array([r["end_position"] for r in reaches])
    i_out_reaches_end_position = get_i_outliers(reaches_end_position)
    # seems a better criterion than the length outliers

    reaches = np.delete(reaches, i_out_reaches_end_position)

    # reaches should last between 0.5 and 4 seconds
    reaches_duration = np.array([r["t_end"] - r["t_beg"] for r in reaches])
    i_out_reaches_duration = [
        i
        for i in range(len(reaches_duration))
        if reaches_duration[i] < 0.5 or reaches_duration[i] > 4
    ]
    reaches = np.delete(reaches, i_out_reaches_duration)

    return reaches, peaks_threshold


def plot_reaches_on_one_wrist(
    ax, t, wrist, wrist_f, reaches, label="wrist", color="b", thresh=None
):
    """ " Plot the reaches of the wrist"""
    ax.plot(t, wrist, ".", label=label, color=color)
    ax.plot(t, wrist_f, label=f"{label} filtered", color=color, alpha=0.2)

    # plot the threshold
    if thresh is not None:
        ax.axhline(
            y=thresh,
            color=color,
            linestyle="--",
            label=f"Threshold {label}",
        )
    for reach in reaches:
        # plot the start and end of the reach + a line
        i_beg = reach["i_beg"]
        i_end = reach["i_end"]
        ax.plot(
            t[i_beg],
            wrist_f[i_beg],
            "o",
            color="orange",
        )
        ax.plot(
            t[i_end],
            wrist_f[i_end],
            "o",
            color="r",
        )
        ax.plot(
            [t[i_beg], t[i_end]],
            [wrist_f[i_beg], wrist_f[i_end]],
            color="k",
            linestyle="--",
        )

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()


def is_reach_file(xdf_fullFname: str | os.PathLike):
    """Check if the xdf file is a reach file"""

    full_fname = str(xdf_fullFname)
    fname = os.path.basename(full_fname)
    if "Reaching" in full_fname:
        return True

    if "_r.xdf" in fname:
        return True

    if "_Reach" in fname:
        return True

    return False


def get_reach_list(reaches_left, reaches_right):
    """Make a single reach list from the left and right reaches, sorted by time"""
    reaches = []
    for reach in reaches_left:
        reach["wrist"] = "left"
        reaches.append(reach)
    for reach in reaches_right:
        reach["wrist"] = "right"
        reaches.append(reach)
    reaches = sorted(reaches, key=lambda x: x["t_beg"])

    # add the index of the reach
    for i in range(len(reaches)):
        reaches[i]["index"] = i

    return reaches

def plot_actions_start_stop_times(ax, actions):
    """Plot the start and stop times of the actions"""

    for i in range(len(actions["start_times"])):
        ax.axvspan(
            actions["start_times"][i],
            actions["stop_times"][i],
            color="gray",
            alpha=0.2,
            label= f"Actions ({actions["stream_name"]})" if i == 0 else "",
        )

def get_reaches_from_both_wrist(xdf_file: XDF_file, actions=None):
    """Get the reaches from both wrists"""

    if not is_reach_file(xdf_file.xdf_fullFname):
        return None

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo:
        stream = xdf_file.streams[i_k_mo]
        kinect_t = stream.time_stamps

        i_wrz = stream.get_channel_index("WristRight_Z")
        i_wlz = stream.get_channel_index("WristLeft_Z")

        WristLeft_xyz = stream.time_series[:, i_wlz : i_wlz + 3]
        WristRight_xyz = stream.time_series[:, i_wrz : i_wrz + 3]

        WristLeft_Norm = np.linalg.norm(WristLeft_xyz, axis=1)
        WristRight_Norm = np.linalg.norm(WristRight_xyz, axis=1)

        WristLeft_Norm_f = lowpass_filter(kinect_t, WristLeft_Norm, cutoff=0.5)
        WristRight_Norm_f = lowpass_filter(kinect_t, WristRight_Norm, cutoff=0.5)

        reaches_left, thresh_left = get_reaches_on_one_wrist(
            kinect_t, WristLeft_Norm, WristLeft_Norm_f
        )
        reaches_right, thresh_right = get_reaches_on_one_wrist(
            kinect_t, WristRight_Norm, WristRight_Norm_f
        )

        reaches = get_reach_list(reaches_left, reaches_right)

        if doRunTests:
            fig, ax = plt.subplots(figsize=(10, 5))
            plot_reaches_on_one_wrist(
                ax,
                kinect_t,
                WristLeft_Norm,
                WristLeft_Norm_f,
                reaches_left,
                label="Left wrist",
                color="b",
                thresh=thresh_left,
            )
            plot_reaches_on_one_wrist(
                ax,
                kinect_t,
                WristRight_Norm,
                WristRight_Norm_f,
                reaches_right,
                label="Right wrist",
                color="k",
                thresh=thresh_right,
            )

            # ### plot the actions start and stop times
            if actions:
                plot_actions_start_stop_times(ax, actions)

            plt.title(
                "Reaches detected from the wrist position (low pass filtered @ 0.5Hz)"
                + f"\n{xdf_file.xdf_fullFname}"
            )

            plt.legend()
            plt.show()

    logging.info(
        f"Found {len(reaches_left)} reaches on the left wrist and {len(reaches_right)} reaches on the right wrist"
    )
    return reaches



def print_reaches(reaches):
    for i, reach in enumerate(reaches):
        print(
            f"{reach["index"]:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']}"
        )


#########################################################################################

if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # Test file for panu identification
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P05/V1/Reaching/C01P05_EttMha_20210621_1_r.xdf"  # Test file for panu identification
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P02/V2/Reaching/002_CorJea_20210409_2_r.xdf"  # 
    xdf_fullFname =  '../dat/ReArm.lnk/DATA_named/C1P03/V2/Reaching/003_DucPas_20210430_2_r.xdf' # a lot of empty values

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    # TODO: check if the data still exists after removing the zeros... 
    resample_kinect_data(xdf_file)

    actions = get_actions_from_markers(xdf_file) # before reaches (used in plotting)
    reaches = get_reaches_from_both_wrist(xdf_file, actions)

    print_reaches(reaches)

## 5.2. <a id='toc5_2_'></a>[Get the conditions from the reaches sequence](#toc0_)

Within one action zone, the participant is asked to perform a series of reaches with one hand (typically a block of 5 reaches).  
When the participant switches from one hand to the other, this is the end of the action zone with this hand. 

The next reach is performed with the other hand in the next action zone.

In [ ]:
def ax_plot_conditions(ax, conditions):
    """Plot the conditions on the ax"""

    def ax_plot_one_condition(conditions, key, color):
        if key in conditions:
            ax.axvspan(
                conditions[key]["t_beg"],
                conditions[key]["t_end"],
                color=color,
                alpha=0.2,
                label=key,
            )

    ax_plot_one_condition(conditions, "sau-p", "orange")
    ax_plot_one_condition(conditions, "sau-np", "purple")
    ax_plot_one_condition(conditions, "mau-p", "red")
    ax_plot_one_condition(conditions, "mau-np", "green")
    ax_plot_one_condition(conditions, "cal", "yellow")


def plot_conditions(reaches, conditions, actions=None):
    """Plot the conditions on the reaches"""

    # plot the reach sequence

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, reach in enumerate(reaches):
        if reach["wrist"] == "left":
            color = "b"
        else:
            color = "k"
        ax.plot(
            [reach["t_beg"], reach["t_end"]],
            [reach["beg_position"], reach["end_position"]],
            color=color,
            linestyle="--",
        )
        ax.plot(
            reach["t_beg"],
            reach["beg_position"],
            "o",
            color=color,
        )
        ax.plot(
            reach["t_end"],
            reach["end_position"],
            "o",
            color=color,
        )

    # plot the conditions
    ax_plot_conditions(ax, conditions)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")

    title_txt = "Conditions from the reach sequence only (no markers needed)"
    actions_txt = (
        "" if actions is None else f", actions (grayed) from {actions['stream_name']}"
    )

    if actions is not None:
        plot_actions_start_stop_times(ax, actions)

    plt.title(title_txt + actions_txt + f"\n{xdf_file.xdf_fullFname}")
    ax.legend()
    plt.show()


def get_conditions(reaches):
    """Get the conditions from the reaches sequence"""

    ####### Get the blocks of reaches
    # in a block, the time between the reaches is less than 10s
    # classically 4-5 reaches in a 20s block, but minimum 2
    reaches_starts = np.array([reach["t_beg"] for reach in reaches])
    intervals = np.diff(reaches_starts)
    in_block_time = intervals < 10

    # in a block, the hand is the same
    reaches_hand = np.array(
        [1 if reach["wrist"] == "right" else 0 for reach in reaches]
    )
    change_hand = np.diff(reaches_hand)
    in_block_hand = change_hand == 0

    # in a block, the hand is the same AND the time between reaches is less than 10s
    in_block = in_block_time & in_block_hand

    # get the changes in block
    change_block = np.diff(in_block.astype(int))

    # the next reach after a change_block is the start-end of a new block
    r_beg_blocks = np.where(change_block == 1)[0] + 1
    r_end_blocks = np.where(change_block == -1)[0] + 1

    # do not forget to add the first and last limits
    if in_block[0]:
        r_beg_blocks = np.insert(r_beg_blocks, 0, 0)
    if in_block[-1]:
        r_end_blocks = np.append(r_end_blocks, len(reaches) - 1)

    ####### Get the limits of conditions (from the changes in hand between block)
    # in a condition, the hand is the same (at least 1 block of one hand)

    hand_blocks = np.array(
        [
            1 if reaches[r_beg_blocks[i]]["wrist"] == "right" else 0
            for i in range(len(r_beg_blocks))
        ]
    )

    # identify the changes in hand from block to block
    change_hand_between_blocks = np.diff(hand_blocks)
    b_change = np.where(change_hand_between_blocks != 0)[0]

    # a beg block is after a change of hand
    b_beg = b_change + 1
    # add the first block to get all blocks of beg of a condition
    b_beg = np.insert(b_beg, 0, 0)

    # a end block is at a change of hand
    b_end = b_change
    # add the last block to get all blocks of end of a condition
    b_end = np.append(b_end, len(hand_blocks) - 1)

    # get the reaches of the beg and end blocks
    r_beg = r_beg_blocks[b_beg]
    r_end = r_end_blocks[b_end]

    # get the time of the beg and end blocks
    t_beg = [reaches[r_beg[i]]["t_beg"] for i in range(len(r_beg))]
    t_end = [reaches[r_end[i]]["t_end"] for i in range(len(r_end))]

    # shift the beg time of 0.5s to compensate for the low pass filter at 0.5Hz
    t_beg = np.array(t_beg) - 0.5
    t_end = np.array(t_end) + 0.5

    ####### Set the conditions
    # the order is sau-p, sau-np, mau-p, mau-np
    rearm_condition_names = ["sau-p", "sau-np", "mau-p", "mau-np"]

    if len(b_beg) != 4:
        logging.warning(
            f"Number of conditions ({len(b_beg)}) is not the expected one in ReArm ({len(rearm_condition_names)})"
        )

    conditions = {}
    for i in range(len(b_beg)):
        # create a dict item for each rearm_condition_names
        conditions[rearm_condition_names[i]] = {
            "name": rearm_condition_names[i],
            "b_beg": b_beg[i],
            "b_end": b_end[i],
            "r_beg": r_beg[i],
            "r_end": r_end[i],
            "t_beg": t_beg[i],
            "t_end": t_end[i],
        }

    ##### The paretic hand is used in the first reach of sau-p
    paretic_side = (
        "left" if reaches[conditions["sau-p"]["r_beg"]]["wrist"] == "left" else "right"
    )

    ####### Add the conditions to the reaches
    for reach in reaches:
        reach["condition"] = "unknown"
        reach["side"] = "paretic" if paretic_side == reach["wrist"] else "non_paretic"
        for condition in conditions:
            if (
                reach["t_beg"] >= conditions[condition]["t_beg"]
                and reach["t_beg"] <= conditions[condition]["t_end"]
            ):
                reach["condition"] = condition.split("-")[0]
                break

    ####### Get the calibration condition
    def first_calibration_reach_pair(reaches_unknown, start_time=None):
        """Get the calibration reach couple from the unknown reaches"""
        # we look for the first two left+right reaches that are less than 30 seconds apart
        cal_start_times = np.array([reach["t_beg"] for reach in reaches_unknown])
        cal_wrists = np.array([reach["wrist"] for reach in reaches_unknown])

        if start_time is None:
            start_time = reaches_unknown[0]["t_beg"]

        # remove the reaches that are before the start time
        cal_start_times = cal_start_times[cal_start_times >= start_time]
        cal_wrists = cal_wrists[cal_start_times >= start_time]

        # retrieve the first left+right pair less than 30 seconds apart
        d_cal_start_times = np.diff(cal_start_times)
        for i in range(len(d_cal_start_times)):
            if d_cal_start_times[i] < 30:
                if cal_wrists[i] != cal_wrists[i + 1]:
                    return (reaches_unknown[i], reaches_unknown[i + 1])
        return None, None

    reaches_unknown = [reach for reach in reaches if reach["condition"] == "unknown"]
    if len(reaches_unknown) > 0:
        reach_left, reach_right = first_calibration_reach_pair(reaches_unknown)
        if reach_left is not None and reach_right is not None:
            first_reach = (
                reach_left
                if reach_left["t_beg"] < reach_right["t_beg"]
                else reach_right
            )
            last_reach = reach_right if first_reach == reach_left else reach_left
            # add the calibration condition
            conditions["cal"] = {
                "name": "cal",
                "b_beg": np.nan,
                "b_end": np.nan,
                "r_beg": first_reach["index"],
                "r_end": last_reach["index"],
                "t_beg": first_reach["t_beg"] - 0.5,
                "t_end": last_reach["t_end"] + 0.5,
            }
            reach_left["condition"] = "calibration"
            reach_right["condition"] = "calibration"

    return conditions


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # Test file for panu identification
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P05/V1/Reaching/C01P05_EttMha_20210621_1_r.xdf"  # Test file for panu identification

    # xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # missing data in the file

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)

    actions = get_actions_from_markers(xdf_file)
    reaches = get_reaches_from_both_wrist(xdf_file, actions)
    conditions = get_conditions(reaches)

    print("Conditions:")
    for item in conditions:
        condition = conditions[item]
        print(
            f"{condition['name']:>8}, block {condition['b_beg']:3.0f} -> {condition['b_end']:3.0f} <=> reach {condition['r_beg']:2d} -> {condition['r_end']:2d}"
        )

    plot_conditions(reaches, conditions, actions=actions)

# 6. <a id='toc6_'></a>[Get target position](#toc0_)
The target position is the position of the mouse at the end of the calibration reaches for the paretic and non-paretic hand.  
The target position is the average of the two positions.

In [ ]:
def get_target_positions(cal, reaches, xdf_file):
    """Get the target positions from the calibration"""

    # identify the calibration reaches left and right
    r1 = cal["r_beg"]
    r2 = cal["r_end"]

    reach_1 = reaches[r1]
    reach_2 = reaches[r2]

    if reach_1["wrist"] == "left":
        reach_left = reach_1
        reach_right = reach_2
    else:
        reach_left = reach_2
        reach_right = reach_1

    # get the kinect mocap stream
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    kinect_mocap = xdf_file.streams[i_k_mo]

    i_wrx = kinect_mocap.get_channel_index("WristRight_X")
    i_wlx = kinect_mocap.get_channel_index("WristLeft_X")

    # get the left and right wrist positions
    wrist_r_xyz = kinect_mocap.time_series[:, i_wrx : i_wrx + 3]
    wrist_l_xyz = kinect_mocap.time_series[:, i_wlx : i_wlx + 3]

    # get the end time of the reach
    i_end_r = reach_right["i_end"]
    i_end_l = reach_left["i_end"]

    # we want the median of the positions at i_data +/- 0.5 seconds
    # we assume that we have enough data before and after the index
    half_window = 15

    mask_r = range(i_end_r - half_window, i_end_r + half_window)
    mask_l = range(i_end_l - half_window, i_end_l + half_window)

    target_r = wrist_r_xyz[mask_r]
    target_l = wrist_l_xyz[mask_l]

    target_r = np.median(target_r, axis=0)
    target_l = np.median(target_l, axis=0)

    # the target is the middle of the left and right positions
    target_xyz = np.mean([target_r, target_l], axis=0)

    return target_xyz


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # Test file for panu identification
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P05/V1/Reaching/C01P05_EttMha_20210621_1_r.xdf"  # Test file for panu identification

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)

    actions = get_actions_from_markers(xdf_file)
    reaches = get_reaches_from_both_wrist(xdf_file, actions)
    conditions = get_conditions(reaches)

    xyz = get_target_positions(conditions["cal"], reaches, xdf_file)

    print(f"Target position: {xyz}")
    # Format the arrays for better readability
    txt = ""
    for x in xyz:
        txt += f"{x:.3f} "
    print(f"target_xyz position (m): {txt}")

# 7. <a id='toc7_'></a>[Compute the distance to the target for both wrists and both shoulders](#toc0_)

In [ ]:
def get_distance_to_target(xyz, target_xyz):
    """Get the distance to the target"""

    # check that the target is a 1D array of 3 elements
    if target_xyz.ndim != 1 or target_xyz.shape[0] != 3:
        raise ValueError("target_xyz must be a 1D array of 3 elements")

    # check that xyz is either a 1D or 2D array of 3 elements by row
    if xyz.ndim == 1:
        # reshape it to a 2D array (with one row)
        xyz = np.array([xyz])
    elif xyz.ndim == 2:
        # check that it has 3 columns
        if xyz.shape[1] != 3:
            raise ValueError("xyz must be a 2D array of 3 elements")
    else:
        raise ValueError("xyz must be a 1D or 2D array")

    # get the distance to the target
    distance_to_target = np.linalg.norm(xyz - target_xyz, axis=1)

    return distance_to_target


def get_joint_xyz(kinect_stream: XDF_stream, joint_name_X: str):
    """Get the joint xyz from the kinect_stream"""

    i_joint = kinect_stream.get_channel_index(joint_name_X)
    if i_joint is None:
        raise ValueError(f"No {joint_name_X} channel found")
    joint_xyz = kinect_stream.time_series[:, i_joint : i_joint + 3]
    return joint_xyz


def get_all_distances_to_target(kinect_stream: XDF_stream, target_xyz):
    """Get the distance to the target for all joints"""

    distances = {
        "raw": {},
        "filtered": {},
    }

    joint_names = [
        "WristRight",
        "WristLeft",
        "ShoulderRight",
        "ShoulderLeft",
    ]

    for joint_name in joint_names:
        joint_xyz = get_joint_xyz(kinect_stream, joint_name + "_X")
        distances["raw"][joint_name] = get_distance_to_target(joint_xyz, target_xyz)
        # NOTE: this "optimal" cutoff was determined by Faity http://doi.org/10.3390/s22072735
        distances["filtered"][joint_name] = lowpass_filter(
            kinect_stream.time_stamps,
            distances["raw"][joint_name],
            cutoff=2.5,
        )

    return distances


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # Test file for panu identification
    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P05/V1/Reaching/C01P05_EttMha_20210621_1_r.xdf"  # Test file for panu identification

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)

    actions = get_actions_from_markers(xdf_file)
    reaches = get_reaches_from_both_wrist(xdf_file, actions)
    conditions = get_conditions(reaches)

    conditions = get_conditions(reaches)

    target_xyz = get_target_positions(conditions["cal"], reaches, xdf_file)

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo is None:
        raise ValueError("No MoCap stream found")

    kinect_stream = xdf_file.streams[i_k_mo]

    distances = get_all_distances_to_target(kinect_stream, target_xyz)

    print("Distances to target:")
    for joint_name, distance in distances.items():
        print(f"{joint_name}: {distance}")

    ## plot the distances
    fig, ax = plt.subplots(figsize=(10, 5))
    for joint_name, distance_values in distances["raw"].items():

        color = "b" if "Left" in joint_name else "k"
        ax.plot(
            kinect_stream.time_stamps,
            distance_values,
            ".",
            label=f"{joint_name} distance",
            markerfacecolor="w" if "Shoulder" in joint_name else color,
            markeredgecolor=color,
            markeredgewidth=0.5 if "Shoulder" in joint_name else 1,
        )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance to target (m)")
    ax.legend()
    plt.title(f"Distances to target from {xdf_file.xdf_fullFname}")
    plt.show()

# 8. <a id='toc8_'></a>[Re-compute the reaches start and stop times from the distance to the target data](#toc0_)

In [ ]:
def get_reach_from_filtered_distance_to_target(reach, kinect_time, distances):
    """Get the reach start and end positions from the filtered distance to target"""

    # if the reach is left, use the left distance
    if reach["wrist"] == "left":
        raw_distance = distances["raw"]["WristLeft"]
        filtered_distance = distances["filtered"]["WristLeft"]
    else:
        raw_distance = distances["raw"]["WristRight"]
        filtered_distance = distances["filtered"]["WristRight"]

    # get the derivative of the filtered distance to target
    filtered_velocity = np.gradient(filtered_distance, kinect_time)

    # Step 0: reach contains the GUESSED reach (from the distance to kinect)
    # Step 1: Get the reach final position after the velocity peak
    # We scout for the minimum distance to target between two velocity peaks:
    #  - from the negative velocity peak before the guessed END
    #    (i.e., the reach-to-target velocity peak)
    #  - to the positive velocity peak after the guessed END
    #    (i.e., the reach-back-from-target velocity peak)
    # Step 2: Get the reach start position before the velocity peak
    # We scout for the maximum distance to target between tow points:
    #  - from the guessed reach START
    #    (which is always too early)
    #  - to the negative velocity peak before the guessed END
    #    (i.e., the reach-to-target velocity peak)

    def get_negative_peak_index(signal, i_beg, i_end):
        """Get the index of the negative peak in a signal between two indices"""
        try:
            i_min = np.argmin(signal[i_beg:i_end]) + i_beg
        except ValueError as e:
            # If the signal is empty or has no valid values, return the beginning index
            print(f"Error in get_negative_peak_index: {e}")
            print(
                f"i_beg: {i_beg}, i_end: {i_end}, signal length: {len(signal)}, reach: {reach['index']}"
            )
            i_min = i_beg
        return int(i_min)

    def get_positive_peak_index(signal, i_beg, i_end):
        """Get the index of the positive peak in a signal between two indices"""
        i_max = np.argmax(signal[i_beg:i_end]) + i_beg
        return int(i_max)

    def get_last_positive_peak_index(signal, i_beg, i_end):
        """Get the index of the last positive peak in a signal between two indices"""

        i_max = np.argmax(signal[i_beg:i_end]) + i_beg
        # find all positive peaks
        pos_peaks, _ = find_peaks(signal[i_beg:i_end])
        if len(pos_peaks) == 0:
            logging.warning(
                f"No positive peaks found between {i_beg} and {i_end} for reach {reach['index']}"
            )
            return int(i_max)
        else:
            i_last_pos_peak = pos_peaks[-1] + i_beg

        return int(i_last_pos_peak)

    # if reach["index"] == 15:
    #     # debug
    #     print("Debug reach 15")

    # 1.1. get the index of the negative velocity peak before the guessed END
    i_neg_vel_peak_before = get_negative_peak_index(
        filtered_velocity,
        reach["i_beg"],
        reach["i_end"],
    )

    # 1.2. get the index of the positive velocity peak after the guessed END
    window_size = 60  # 2 s = typical inter-reach time
    i_pos_vel_peak_after = get_positive_peak_index(
        filtered_velocity,
        reach["i_beg"],
        reach["i_end"] + window_size,
    )
    # As we have the two velocity peaks, we can get the reach end position...
    i_end_reach = get_negative_peak_index(
        filtered_distance,
        i_neg_vel_peak_before,
        i_pos_vel_peak_after,
    )

    # ... and the reach start position
    i_search_beg = reach["i_beg"]
    i_start_reach = get_last_positive_peak_index(
        filtered_distance,
        i_search_beg,
        i_neg_vel_peak_before,
    )

    to_return = {
        "i_beg": i_start_reach,
        "i_end": i_end_reach,
        "t_beg": kinect_time[i_start_reach],
        "t_end": kinect_time[i_end_reach],
    }

    do_plot_debug = False
    if do_plot_debug and reach["index"] in [15, 16]:
        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(
            kinect_time,
            raw_distance,
            ".",
            label="Raw distance to target",
            color="k",
        )

        ax.plot(
            kinect_time,
            filtered_distance,
            ".-",
            linewidth=0.5,
            markersize=1,
            label="Filtered distance to target",
            color="b",
        )
        ax.plot(
            kinect_time,
            filtered_velocity,
            ".-",
            linewidth=0.25,
            markersize=1,
            label="Derivative of distance to target",
            color="r",
        )
        # plot the reach start
        ax.axvline(
            kinect_time[i_start_reach],
            color="g",
            linestyle="--",
        )

        ax.plot(
            kinect_time[i_start_reach],
            filtered_distance[i_start_reach],
            "o",
            label="new Reach start",
            color="g",
        )

        # plot the reach end
        ax.axvline(
            kinect_time[i_end_reach],
            color="r",
            linestyle="--",
        )

        ax.plot(
            kinect_time[i_end_reach],
            filtered_distance[i_end_reach],
            "o",
            label="new Reach end",
            color="r",
        )

        # vertical grey line at i_beg and i_end
        ax.axvline(
            kinect_time[reach["i_beg"]],
            color="grey",
            linestyle="--",
            label="OLD Reach start",
        )
        ax.axvline(
            kinect_time[reach["i_end"]],
            color="grey",
            linestyle="--",
            label="OLD Reach end",
        )

        ax.axvspan(
            kinect_time[(i_search_beg)],
            kinect_time[i_neg_vel_peak_before],
            color="g",
            alpha=0.05,
            label="BEG search window ",
        )
        ax.axvspan(
            kinect_time[(i_neg_vel_peak_before)],
            kinect_time[i_pos_vel_peak_after],
            color="r",
            alpha=0.05,
            label="END search window ",
        )

        # horizontal line at 0
        ax.axhline(0, color="k", linestyle="--")

        # set the x limits to the reach time
        ax.set_xlim(reach["t_beg"] - 10, reach["t_end"] + 10)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Distance to target (m)")
        ax.legend()
        plt.show()

    return to_return


def get_distance(signal, i_beg, i_end):
    """Get the distance between two indexes in a signal"""
    travel = np.abs(signal[i_beg] - signal[i_end])
    return travel


def get_reaches_panu(reaches, kinect_stream, distances, actions):
    """Get the reaches for panu from the filtered distance to target"""

    # get the reach from the filtered distance to target
    reaches_panu = []
    for reach in reaches:
        # get the reach start and end
        limits = get_reach_from_filtered_distance_to_target(
            reach,
            kinect_stream.time_stamps,
            distances,
        )

        if reach["wrist"] == "left":
            w_distance = distances["filtered"]["WristLeft"]
            s_distance = distances["filtered"]["ShoulderLeft"]
        else:
            w_distance = distances["filtered"]["WristRight"]
            s_distance = distances["filtered"]["ShoulderRight"]

        w_travel = get_distance(
            w_distance,
            limits["i_beg"],
            limits["i_end"],
        )
        s_travel = get_distance(
            s_distance,
            limits["i_beg"],
            limits["i_end"],
        )

        reach_panu = {
            "index": reach["index"],
            "wrist": reach["wrist"],
            "condition": reach["condition"],
            "side": reach["side"],
            "i_beg": limits["i_beg"],
            "i_end": limits["i_end"],
            "time_beg": limits["t_beg"],
            "time_end": limits["t_end"],
            "wrist_beg": w_distance[limits["i_beg"]],
            "wrist_end": w_distance[limits["i_end"]],
            "shoulder_beg": s_distance[limits["i_beg"]],
            "shoulder_end": s_distance[limits["i_end"]],
            "wrist_travel": w_travel,
            "shoulder_travel": s_travel,
        }
        # find the corresponding action
        reach_panu["action"] = np.nan
        # if the reach is in the action time, add the action
        for i in range(len(actions["start_times"])):
            if (
                limits["t_beg"] >= actions["start_times"][i]
                and limits["t_beg"] <= actions["stop_times"][i]
            ):
                reach_panu["action"] = i
                break
        # if the reach is within an action or if in calibration it is OK
        if reach_panu["action"] is not np.nan:
            reaches_panu.append(reach_panu)
        if reach_panu["condition"] == "calibration":
            reaches_panu.append(reach_panu)
    return reaches_panu


#######################################################################################
if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # Test file for panu identification
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P05/V1/Reaching/C01P05_EttMha_20210621_1_r.xdf"  # Test file for panu identification

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)

    actions = get_actions_from_markers(xdf_file)
    reaches = get_reaches_from_both_wrist(xdf_file, actions)
    conditions = get_conditions(reaches)

    target_xyz = get_target_positions(conditions["cal"], reaches, xdf_file)

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo is None:
        raise ValueError("No MoCap stream found")

    kinect_stream = xdf_file.streams[i_k_mo]

    distances = get_all_distances_to_target(kinect_stream, target_xyz)

    # get the reaches for panu
    reaches_panu = get_reaches_panu(reaches, kinect_stream, distances, actions)

# 9. <a id='toc9_'></a>[Save the reaches in a csv file](#toc0_)

In [ ]:
def get_reaches_csv_file_name(xdf_fullFname):
    """Get the reaches CSV file name corresponding to the xdf file name"""
    return xdf_fullFname.replace(".xdf", "_reaches_panu.csv")


def save_reaches_in_csv_file_name(reaches, xdf_fullFname):
    """Save the reaches in a CSV file with the file name"""
    file_name = get_reaches_csv_file_name(xdf_fullFname)
    df = pd.DataFrame(reaches)
    df.to_csv(file_name, index=False)
    return file_name


def load_reaches_from_csv_file_name(csv_fullFname):
    """Load the reaches from a CSV file with the file name"""
    df = pd.read_csv(csv_fullFname)
    return df


if doRunTests:
    fname_csv = save_reaches_in_csv_file_name(reaches_panu, xdf_fullFname)
    print(f"'reaches_panu' saved in {fname_csv}")

    reaches_panu_df = load_reaches_from_csv_file_name(fname_csv)
    print(f"'reaches_panu_df' loaded from {fname_csv}")

# 10. <a id='toc10_'></a>[Compute the panu ](#toc0_)

In [ ]:
def compute_panu(xdf_fullFname):
    """Compute the panu for the xdf file"""
    # load the dataframe from the CSV file
    csv_fullFname = get_reaches_csv_file_name(xdf_fullFname)
    if not os.path.exists(csv_fullFname):
        raise ValueError(f"File {csv_fullFname} does not exist")

    df = load_reaches_from_csv_file_name(csv_fullFname)

    # get the median of the ratio  by condition
    paretic_reaches = df[df["side"] == "paretic"]
    sau_reaches = paretic_reaches[paretic_reaches["condition"] == "sau"]
    mau_reaches = paretic_reaches[paretic_reaches["condition"] == "mau"]
    sau_ratio = 1 - sau_reaches["shoulder_travel"] / sau_reaches["wrist_travel"]
    mau_ratio = 1 - mau_reaches["shoulder_travel"] / mau_reaches["wrist_travel"]
    sau_median = sau_ratio.median()
    mau_median = mau_ratio.median()

    panu = mau_median - sau_median

    return {
        "fname": xdf_fullFname,
        "panu": panu,
        "sau": sau_median,
        "mau": mau_median,
    }


if doRunTests:
    xdf_fullFname = (
        "../dat/ReArm.lnk/DATA_Named/C1P05/V1/Reaching/C01P05_EttMha_20210621_1_r.xdf"
    )
    panu = compute_panu(xdf_fullFname)

    print("Panu from CSV file:")
    for key, value in panu.items():
        print(f"{key:>5}: {value:.2f}" if key != "fname" else f"{key}: {value}")

# 11. <a id='toc11_'></a>[plot the reaches used to compute the panu](#toc0_)

In [ ]:
def ax_plot_sau_mau_cal(ax, conditions):
    """Plot the conditions on the ax"""
    if len(conditions) == 0:
        return

    # if sau exists
    if "sau-p" in conditions and "sau-np" in conditions:
        # plot sau
        ax.axvspan(
            conditions["sau-p"]["t_beg"],
            conditions["sau-np"]["t_end"],
            color="blue",
            alpha=0.1,
            label="sau",
        )

    # if mau exists
    if "mau-p" in conditions and "mau-np" in conditions:
        ax.axvspan(
            conditions["mau-p"]["t_beg"],
            conditions["mau-np"]["t_end"],
            color="purple",
            alpha=0.1,
            label="mau",
        )
    # if cal exists
    if "cal" in conditions:
        ax.axvspan(
            conditions["cal"]["t_beg"],
            conditions["cal"]["t_end"],
            color="red",
            alpha=0.1,
            label="cal",
        )


def plot_panu(
    xdf_fullFname,
    kinect_stream,
    distances,
    reaches_panu,
    conditions,
    panu
):

    # create the figure
    fig_panu, ax = plt.subplots(figsize=(30, 15))

    # shortcut for time
    t = kinect_stream.time_stamps

    # plot the distances to target
    for joint_name, distance_values in distances["raw"].items():
        color = "b" if "Left" in joint_name else "k"
        ax.plot(
            t,
            distance_values,
            ".",
            label=f"{joint_name} distance",
            markerfacecolor="w" if "Shoulder" in joint_name else color,
            markeredgecolor=color,
            markeredgewidth=0.5 if "Shoulder" in joint_name else 1,
        )

    # plot the filtered distances to target
    for joint_name, distance_values in distances["filtered"].items():
        color = "b" if "Left" in joint_name else "k"
        ax.plot(
            t,
            distance_values,
            ".-",
            # label=f"{joint_name} distance (filtered)",
            markerfacecolor="w" if "Shoulder" in joint_name else color,
            markeredgecolor=color,
            markeredgewidth=0.5 if "Shoulder" in joint_name else 1,
            linewidth=0.25,
            markersize=1,
            color=color,
        )
    # plot the conditions
    ax_plot_sau_mau_cal(ax, conditions)

    # plot the reaches
    for reach in reaches_panu:
        if reach["wrist"] == "left":
            color = "b"
            w_distance = distances["filtered"]["WristLeft"]
            s_distance = distances["filtered"]["ShoulderLeft"]
        else:
            color = "k"
            w_distance = distances["filtered"]["WristRight"]
            s_distance = distances["filtered"]["ShoulderRight"]

        ax.plot(
            t[reach["i_beg"]],
            w_distance[reach["i_beg"]],
            "o",
            color="orange",
        )
        ax.plot(
            t[reach["i_end"]],
            w_distance[reach["i_end"]],
            "o",
            color="r",
        )

        ax.plot(
            t[reach["i_beg"]],
            s_distance[reach["i_beg"]],
            "o",
            color="orange",
        )
        ax.plot(
            t[reach["i_end"]],
            s_distance[reach["i_end"]],
            "o",
            color="r",
        )

    ylim = ax.get_ylim()
    ax.set_ylim(0, ylim[1])
    ax.legend()
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance to target (m)")

    plt.title(
        f"{xdf_fullFname}: PANU: {panu["panu"]:.2f}, SAU: {panu["sau"]:.2f}, MAU: {panu["mau"]:.2f}"
    )
    plt.tight_layout()

    return fig_panu


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # Test file for panu identification
    #xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P05/V1/Reaching/C01P05_EttMha_20210621_1_r.xdf"  # Test file for panu identification
    #xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P02/V2/Reaching/002_CorJea_20210409_2_r.xdf"  # 

    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P03/V1/Reaching/003_DucPas_20210412_1_r.xdf"

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo is None:
        raise ValueError("No MoCap stream found")

    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)

    actions = get_actions_from_markers(xdf_file)
    reaches = get_reaches_from_both_wrist(xdf_file, actions)
    conditions = get_conditions(reaches)

    target_xyz = get_target_positions(conditions["cal"], reaches, xdf_file)

    kinect_stream = xdf_file.streams[i_k_mo]
    distances = get_all_distances_to_target(kinect_stream, target_xyz)
    reaches_panu = get_reaches_panu(reaches, kinect_stream, distances, actions)
    panu=compute_panu(xdf_fullFname)

    fig_panu = plot_panu(
        xdf_fullFname,
        kinect_stream,
        distances,
        reaches_panu,
        conditions,
        panu,
    )

    save_reaches_in_csv_file_name(reaches_panu, xdf_fullFname)
    print(f"'reaches_panu' saved in {xdf_fullFname.replace('.xdf', '_reaches.csv')}")

    fig_fullFname = xdf_fullFname.replace(".xdf", "_xdf_PANU.png")
    fig_panu.savefig(fig_fullFname, dpi=300)
    print(f"Figure saved in {fig_fullFname}")

# 12. <a id='toc12_'></a>[Compute panu for one xdf file](#toc0_)


In [ ]:
def save_panu(xdf_fullFname):

    # load the xdf file
    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo is None:
        raise ValueError("No MoCap stream found")
    kinect_stream = xdf_file.streams[i_k_mo]

    # order matters !
    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)

    actions = get_actions_from_markers(xdf_file)
    reaches = get_reaches_from_both_wrist(xdf_file, actions)
    conditions = get_conditions(reaches)

    # if the condition cal is not present, we cannot compute the panu
    if "cal" not in conditions:
        logging.warning("No calibration condition found. Cannot compute panu.")
        return

    target_xyz = get_target_positions(conditions["cal"], reaches, xdf_file)

    distances = get_all_distances_to_target(kinect_stream, target_xyz)

    reaches_panu = get_reaches_panu(reaches, kinect_stream, distances, actions)
    save_reaches_in_csv_file_name(reaches_panu, xdf_fullFname)
    panu = compute_panu(xdf_fullFname)

    fig_panu = plot_panu(
        xdf_fullFname, kinect_stream, distances, reaches_panu, conditions, panu
    )
    fig_fullFname = xdf_fullFname.replace(".xdf", "_xdf_PANU.png")
    fig_panu.savefig(fig_fullFname, dpi=300)
    logging.info(f"Figure saved in {fig_fullFname}")

    # save the panu in a CSV file
    panu_fullFname = xdf_fullFname.replace(".xdf", "_xdf_panu.csv")
    panu_df = pd.DataFrame([panu])
    panu_df.to_csv(panu_fullFname, index=False)
    logging.info(f"Saved '{panu_fullFname}'")

# 13. <a id='toc13_'></a>[Compute panu for one visit](#toc0_)

In [ ]:
def get_xdf_files_in_visit(visit_dir, directories_to_skip=None):
    """Get the xdf files in the visit_dir"""

    xdf_files = []

    if not os.path.exists(visit_dir):
        raise ValueError(f"Directory {visit_dir} does not exist")

    for root, dirs, files in os.walk(visit_dir):
        # Skip the directories that are in the directories_to_skip list
        if directories_to_skip and any(
            skip_dir in root for skip_dir in directories_to_skip
        ):
            continue
        for file in files:
            if directories_to_skip and any(
                skip_dir in root for skip_dir in directories_to_skip
            ):
                continue
            if file.endswith(".xdf"):
                xdf_files.append(os.path.join(root, file))

    if not xdf_files:
        logging.warning(f"No xdf files found in {visit_dir}")

    return xdf_files


def is_already_done_panu_in_visit(visitPath, checkLog_fname):
    """
    Check if the visit was already processed with panu
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)
    return os.path.isfile(full_checkLog_fname)


def create_panu_log_file(visitPath, checkLog_fname):
    """
    Create the log file for panu
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)

    # Create the log file
    logging.basicConfig(
        filename=full_checkLog_fname,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True,  # remove previous handlers and set the new one
    )

    return full_checkLog_fname


def merge_panu_png_files_to_pdf(visit_path):
    """
    Merge the panu png files in the visit folder
    """
    png_files = [f for f in os.listdir(visit_path) if f.endswith("_panu.png")]
    png_files.sort()

    if len(png_files) > 1:
        # read the png files
        images = [
            Image.open(os.path.join(visit_path, png_file)) for png_file in png_files
        ]
        # convert to RGB
        images = [img.convert("RGB") for img in images]

        images[0].save(
            os.path.join(visit_path, f"{os.path.basename(visit_path)}_panu_png.pdf"),
            save_all=True,
            append_images=images[1:],
        )

        # remove the original png files
        for png_file in png_files:
            os.remove(os.path.join(visit_path, png_file))
            # print(f"    Removed {png_file}")
    else:
        msg = "No panu png files to merge"
        logging.info(msg)
        print(msg)


def merge_panu_pdf_files_to_pdf(visit_path):
    """
    Merge the panu pdf files in the visit folder
    """
    pdf_files = [f for f in os.listdir(visit_path) if f.endswith("_panu.pdf")]
    pdf_files.sort()

    from pypdf import PdfWriter

    if len(pdf_files) > 1:
        # read the pdf files
        pdf_merger = PdfWriter()
        for pdf_file in pdf_files:
            pdf_merger.append(os.path.join(visit_path, pdf_file))

        pdf_merger.write(
            os.path.join(visit_path, f"{os.path.basename(visit_path)}_panu_pdf.pdf")
        )
        pdf_merger.close()

        # remove the original pdf files
        for pdf_file in pdf_files:
            os.remove(os.path.join(visit_path, pdf_file))
            # print(f"    Removed {pdf_file}")
            pass
    else:
        msg = "No panu pdf files to merge"
        logging.info(msg)
        print(msg)


def get_panu_in_visit(visit_dir, directories_to_skip=None):
    """Correct the kinect timestamps for all the xdf files in the visit_dir"""

    panu_log = "panu.log"
    xdf_files = get_xdf_files_in_visit(visit_dir, directories_to_skip)

    if not xdf_files or len(xdf_files) == 0:
        return

    if is_already_done_panu_in_visit(visit_dir, panu_log):
        print(f"    Already done: '{panu_log}' found")
        return

    create_panu_log_file(visit_dir, panu_log)
    logging.info(f"Starting panu in {visit_dir}")

    for xdf_fullFname in xdf_files:
        if is_reach_file(xdf_fullFname):
            print(f"---- \n{xdf_fullFname}")
            logging.info(f"{os.path.basename(xdf_fullFname)}")
            save_panu(xdf_fullFname)

    merge_panu_png_files_to_pdf(os.path.dirname(visit_dir))
    logging.info("panu completed")
    print(f"    panu completed: see '{panu_log}' for details")


if doRunTests:

    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P02/V2"

    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P01/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P21/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P23/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P20/V2"

    #
    #
    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P38/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P07/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P05/V1"
    os.remove(os.path.join(visit_dir, "panu.log"))
    get_panu_in_visit(visit_dir, directories_to_skip=["old", "Training", "Armeo"])